# 01 – Datenexploration
BPI Challenge 2017: Kreditanträge eines niederländischen Finanzinstituts (2016).
Ziel: Daten kennenlernen, bevor wir den Prozess analysieren.

In [12]:
from pathlib import Path

import pandas as pd

DATA_FILE = Path("..") / "data" / "processed" / "bpi2017.parquet"
df = pd.read_parquet(DATA_FILE)

In [13]:
df.head(10)

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID
0,Created,User_1,A_Create Application,Application,Application_652823628,complete,2016-01-01 09:51:15.304+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,None,NaN,None,NaN,NaN,NaN
1,statechange,User_1,A_Submitted,Application,ApplState_1582051990,complete,2016-01-01 09:51:15.352+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,None,NaN,None,NaN,NaN,NaN
2,Created,User_1,W_Handle leads,Workflow,Workitem_1298499574,schedule,2016-01-01 09:51:15.774+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,None,NaN,None,NaN,NaN,NaN
3,Deleted,User_1,W_Handle leads,Workflow,Workitem_1673366067,withdraw,2016-01-01 09:52:36.392+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,None,NaN,None,NaN,NaN,NaN
4,Created,User_1,W_Complete application,Workflow,Workitem_1493664571,schedule,2016-01-01 09:52:36.403+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,None,NaN,None,NaN,NaN,NaN
5,statechange,User_1,A_Concept,Application,ApplState_642383566,complete,2016-01-01 09:52:36.413+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,None,NaN,None,NaN,NaN,NaN
6,Obtained,User_17,W_Complete application,Workflow,Workitem_1875340971,start,2016-01-02 10:45:22.429+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,None,NaN,None,NaN,NaN,NaN
7,Released,User_17,W_Complete application,Workflow,Workitem_1452291795,suspend,2016-01-02 10:49:28.816+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,None,NaN,None,NaN,NaN,NaN
8,statechange,User_52,A_Accepted,Application,ApplState_99568828,complete,2016-01-02 11:23:04.299+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,None,NaN,None,NaN,NaN,NaN
9,Created,User_52,O_Create Offer,Offer,Offer_148581083,complete,2016-01-02 11:29:03.994+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,20000.0,44.0,True,498.29,True,979.0,20000.0,NaN


In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1202267 entries, 0 to 1202266
Data columns (total 19 columns):
 #   Column                 Non-Null Count    Dtype              
---  ------                 --------------    -----              
 0   Action                 1202267 non-null  str                
 1   org:resource           1202267 non-null  str                
 2   concept:name           1202267 non-null  str                
 3   EventOrigin            1202267 non-null  str                
 4   EventID                1202267 non-null  str                
 5   lifecycle:transition   1202267 non-null  str                
 6   time:timestamp         1202267 non-null  datetime64[us, UTC]
 7   case:LoanGoal          1202267 non-null  str                
 8   case:ApplicationType   1202267 non-null  str                
 9   case:concept:name      1202267 non-null  str                
 10  case:RequestedAmount   1202267 non-null  float64            
 11  FirstWithdrawalAmount  42995 non-nu

In [15]:
offer_rows = df[df["OfferedAmount"].notna()]
offer_rows["concept:name"].value_counts()

concept:name
O_Create Offer    42995
Name: count, dtype: int64

In [16]:
n_missing = df["OfferedAmount"].isna().sum()
n_create_offer = (df["concept:name"] == "O_Create Offer").sum()
n_missing == len(df) - n_create_offer

np.True_

In [17]:
activity_counts = df["concept:name"].value_counts()
print(f"Anzahl verschiedener Aktivitäten: {len(activity_counts)}")
activity_counts


Anzahl verschiedener Aktivitäten: 26


concept:name
W_Validate application        209496
W_Call after offers           191092
W_Call incomplete files       168529
W_Complete application        148900
W_Handle leads                 47264
O_Create Offer                 42995
O_Created                      42995
O_Sent (mail and online)       39707
A_Validating                   38816
A_Create Application           31509
A_Concept                      31509
A_Accepted                     31509
A_Complete                     31362
O_Returned                     23305
A_Incomplete                   23055
O_Cancelled                    20898
A_Submitted                    20423
O_Accepted                     17228
A_Pending                      17228
A_Cancelled                    10431
O_Refused                       4695
A_Denied                        3753
W_Assess potential fraud        3282
O_Sent (online only)            2026
W_Shortened completion           238
W_Personal Loan collection        22
Name: count, dtype: int64

In [18]:
candidates = ["A_Accepted", "A_Pending", "A_Denied", "A_Cancelled"]
for activity in candidates:
    n = df.loc[df["concept:name"] == activity, "case:concept:name"].nunique()
    print(f"{activity:12} kommt in {n:>6,} Anträgen vor")
    

A_Accepted   kommt in 31,509 Anträgen vor
A_Pending    kommt in 17,228 Anträgen vor
A_Denied     kommt in  3,752 Anträgen vor
A_Cancelled  kommt in 10,431 Anträgen vor


In [19]:
OUTCOMES = {"A_Pending": "approved", "A_Denied": "denied", "A_Cancelled": "cancelled"}

outcome_events = df[df["concept:name"].isin(OUTCOMES)]
outcome = outcome_events.groupby("case:concept:name")["concept:name"].first().map(OUTCOMES)
outcome.head()


case:concept:name
Application_1000086665    cancelled
Application_1000158214     approved
Application_1000311556    cancelled
Application_1000334415       denied
Application_1000339879     approved
Name: concept:name, dtype: str

In [20]:
cases = df.groupby("case:concept:name")[
    ["case:LoanGoal", "case:ApplicationType", "case:RequestedAmount"]
].first()
cases["outcome"] = outcome
cases["outcome"] = cases["outcome"].fillna("open")
cases["outcome"].value_counts(normalize=True).round(3)


outcome
approved     0.547
cancelled    0.331
denied       0.119
open         0.003
Name: proportion, dtype: float64

In [21]:
cancel_events = df[df["concept:name"] == "A_Cancelled"]
cancel_events["org:resource"].value_counts(normalize=True).head(5).round(3)


org:resource
User_1     0.762
User_5     0.013
User_3     0.012
User_2     0.010
User_49    0.009
Name: proportion, dtype: float64

In [22]:
cancelled_ids = cancel_events["case:concept:name"]
events_cancelled = df[df["case:concept:name"].isin(cancelled_ids)]

sent_events = events_cancelled[events_cancelled["concept:name"].str.startswith("O_Sent")]
last_sent = sent_events.groupby("case:concept:name")["time:timestamp"].max()
cancelled_at = cancel_events.groupby("case:concept:name")["time:timestamp"].first()

days_until_cancel = (cancelled_at - last_sent).dt.total_seconds() / 86400
days_until_cancel.describe().round(1)


count    10321.0
mean        27.0
std         10.4
min          0.0
25%         30.5
50%         30.7
75%         30.8
max        167.2
Name: time:timestamp, dtype: float64

#### Findings:

Ein drittel der Kredit anträge wird nach 30 tagen abgebrochen. Die weißt darauf hin dass eien Policy besteht welche dem Kunden 30 Tage lässt, um das kreditangebot anzunehmen. 